In [ ]:
import gymnasium as gym

from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env

from stable_baselines3.common.env_checker import check_env

from modules.reinforcement_learning.env import EEGOpt

In [ ]:
import pandas as pd
from modules.load_config import load_params_from_yaml

import numpy as np
from datetime import datetime, timedelta
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import base64

from modules.pf_calculations import gini, apply_pf_schedule_to_mps, plot_stacked_gain_loss_sortable, print_baseline_energy_data, print_gini_on_smp_energy_flow, print_opt_energy_data
from modules.visualisations import plot_profile_by_category,plot_distribution_comparison,plot_sorted_mps_comparison

from plotly.io import to_html
from IPython.display import display, HTML

from modules.optimization_algorithms.EqualWaterfillingOptAlgo import EqualWaterfillingOptAlgo
from modules.optimization_algorithms.OptAlgo import OptAlgo
from datetime import datetime, UTC

from pathlib import Path
import json


In [ ]:
# # PARMS
# changeable
consumer_org_ids = [1] # [1, 2]
# prosumer_org_ids = [13] # [13, 17]

start_time = datetime(2025, 6, 21)
end_time = datetime(2025, 6, 23)

# CONFIG FILE
path_to_local_data = "../../config/"
notebooks_config_file = "notebooks.yaml"

In [ ]:
config_dict = load_params_from_yaml([f"{path_to_local_data}{notebooks_config_file}"])
path_to_local_data = config_dict["PATH_TO_LOCAL_DATA"]
input_filename_template = config_dict["SINGLE_MPS_OF_EEG"]

In [ ]:
raw_eeg = []

for eeg_type, eeg_ids_of_type in [
    ("c", consumer_org_ids),
    # ("p", prosumer_org_ids),
]:
    for act_org_id in eeg_ids_of_type:
        act_filepath_to_load = (
            f"{path_to_local_data}{input_filename_template.format(org_id=act_org_id)}"
        )
        act_df = pd.read_csv(act_filepath_to_load)
        act_df["eeg_type"] = eeg_type
        raw_eeg.append(act_df)

# row-wise append into a single DataFrame
raw_eeg_df = pd.concat(raw_eeg, ignore_index=True)

raw_eeg_df['time'] = pd.to_datetime(raw_eeg_df['time'], utc=True)

eeg_selected_time_horizon = raw_eeg_df[
    (raw_eeg_df["time"] > pd.Timestamp(start_time, tz='UTC')) &
    (raw_eeg_df["time"] < pd.Timestamp(end_time, tz='UTC'))
]
del raw_eeg_df



print(f"{eeg_selected_time_horizon.dtypes}")
print(f"len: {len(eeg_selected_time_horizon)}")
eeg_selected_feat = eeg_selected_time_horizon[["time", "organization_id", "metering_point_id", "energy_direction", "wt_meas_cons", "comm_pot", "comm_cov", "wt_meas_gen", "wt_surp_gen"]].copy()
del eeg_selected_time_horizon

# cons_gen: Consumed Generation, how much of the generated electricity was consumed within the EEG
eeg_selected_feat.sum(numeric_only=True)

# PRINT DF INFO
# 
print(eeg_selected_feat.dtypes)
print(f"len: {len(eeg_selected_feat)}")
print(eeg_selected_feat.drop(["organization_id", "metering_point_id"], axis=1).describe())
eeg_selected_feat.head()

#
# FEATURE ENGINEERIN
#

eeg_selected_feat["cons_gen"] = eeg_selected_feat["wt_meas_gen"] - eeg_selected_feat["wt_surp_gen"]

In [ ]:
work = eeg_selected_feat.copy()

In [ ]:
mp_counts_on_time = (
    work
    .groupby("time")["energy_direction"]
    .value_counts()
    .unstack(fill_value=0)
    .rename(columns={"C": "count_C_mps", "G": "count_G_mps"})
)
sums_on_time = work.groupby(by="time").sum().reset_index().drop(columns=["metering_point_id", "energy_direction"]).rename(columns={"wt_meas_cons":"sum_wt_meas_cons", "comm_pot":"sum_comm_pot", "comm_cov":"sum_comm_cov", "wt_meas_gen":"sum_wt_meas_gen", "wt_surp_gen":"sum_wt_surp_gen", "cons_gen":"sum_cons_gen"})

agg_on_time = pd.merge(left=sums_on_time, right=mp_counts_on_time, on="time", how="outer")
agg_on_time[["time", "count_C_mps", "count_G_mps"]].describe()

In [ ]:
work

In [ ]:
from stable_baselines3.common.vec_env import SubprocVecEnv

# wrapper for env instances
def make_env():
    return lambda: EEGOpt(work)

# 4 parallel envs
num_envs = 12
envs = SubprocVecEnv([make_env() for _ in range(num_envs)])

# PPO with vectorized env
model = PPO("MlpPolicy", envs, device="cpu", verbose=1)
model.learn(total_timesteps=250)


env = EEGOpt(work)

model = PPO("MlpPolicy", env, verbose=1)
model.learn(total_timesteps=25000)
model.save("test")


#del model # remove to demonstrate saving and loading

model = PPO.load("own_tic-tac-toe")

obs, info = env.reset()
while True:
    action, _states = model.predict(obs)
    obs, rewards, dones, truncated,  info = env.step(action)
    print(env.print_state())
    if dones:   
        obs, info=env.reset()
    input()

In [ ]:
env = EEGOpt(work)
obs, info = env.reset()
i=0
while i < 100:
    i += 1
    action, _states = model.predict(obs)
    obs, rewards, dones, truncated,  info = env.step(action)
    if dones:   
        obs, info=env.reset()
    print(rewards)
     
env.act_episode_ed

In [ ]:
applied_pfs = env.act_episode_ed




In [ ]:


# 2) Filter dt by this value
single_time_filtered = applied_pfs
check_full_calc_via_single_timestamp = (
    single_time_filtered[single_time_filtered["energy_direction"] == "C"]
    .groupby(by="metering_point_id")
    .sum(numeric_only=True)[
        [
        "a_opt",
        "comm_cov",
        
    ]
    ]
).add_prefix("sum_")

In [ ]:
check_full_calc_via_single_timestamp

In [ ]:
plot_stacked_gain_loss_sortable(     
check_full_calc_via_single_timestamp, "sum_comm_cov", "sum_a_opt")